In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tarfile
import os

file_path = "/content/drive/MyDrive/sentences.tar.bz2"
extraction_path = "/content/drive/MyDrive/sentences_extracted" # Define the extraction path

# Create the extraction directory if it doesn't exist
if not os.path.exists(extraction_path):
    os.makedirs(extraction_path)

with tarfile.open(file_path, "r:bz2") as tar:
    tar.extractall(path=extraction_path) # Use the extraction_path

print(os.listdir(extraction_path)[:10]) # List files in the extraction directory


/tmp/ipython-input-2137622205.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extraction_path) # Use the extraction_path


['sentences.csv']


In [ ]:
import pandas as pd

sent_file = "/content/drive/MyDrive/sentences_extracted/sentences.csv"  # or sentences.tsv

df = pd.read_csv(sent_file, sep="\t", header=None, names=["id", "lang", "text"])
print(df.head())
print(df.shape)
top_langs = df["lang"].value_counts().head(10)
print(top_langs)


   id lang                    text
0   1  cmn                  我們試試看！
1   2  cmn                 我该去睡觉了。
2   3  cmn                 你在干什麼啊？
3   4  cmn                  這是什麼啊？
4   5  cmn  今天是６月１８号，也是Muiriel的生日！
(12995545, 3)
lang
eng    1991558
rus    1153958
ita     943227
epo     797760
kab     771733
tur     739721
deu     735472
ber     693195
fra     687622
por     437195
Name: count, dtype: int64


In [ ]:
!pip install lightgbm

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import lightgbm as lgb
from sklearn.naive_bayes import MultinomialNB

# ==== 1. Load data ====
sent_file = "/content/drive/MyDrive/sentences_extracted/sentences.csv"
df = pd.read_csv(sent_file, sep="\t", header=None, names=["id", "lang", "text"])

# ==== 2. Chọn subset ngôn ngữ ====
langs = ['eng', 'rus', 'ita', 'epo', 'kab', 'tur', 'deu', 'ber', 'fra', 'por']
df = df[df.lang.isin(langs)]

# ==== 3. Cân bằng dữ liệu (mỗi ngôn ngữ lấy số câu bằng ngôn ngữ ít nhất) ====
min_count = df.lang.value_counts().min()
print("Số mẫu mỗi ngôn ngữ:", min_count)

df_balanced = (
    df.groupby("lang", group_keys=False)
      .apply(lambda x: x.sample(n=min_count, random_state=42))
      .reset_index(drop=True)
)

print(df_balanced.lang.value_counts())
print("Balanced shape:", df_balanced.shape)

# ==== 4. Train-test split ====
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced["text"], df_balanced["lang"],
    test_size=0.2, random_state=42, stratify=df_balanced["lang"]
)

# ==== 5. Pipeline TF-IDF + LightGBM ====
lgbm_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2))),
    ("lgbm", lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(langs),
        boosting_type="gbdt",
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=64,
        random_state=42
    ))
])

# ==== 6. Pipeline TF-IDF + Naive Bayes ====
nb_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2))),
    ("nb", MultinomialNB())
])

# ==== 7. Train + Evaluate LightGBM ====
print("\n=== LightGBM Training ===")
lgbm_model.fit(X_train, y_train)
y_pred_lgbm = lgbm_model.predict(X_test)
print("LightGBM Accuracy:", accuracy_score(y_test, y_pred_lgbm))
print(classification_report(y_test, y_pred_lgbm))

# ==== 8. Train + Evaluate Naive Bayes ====
print("\n=== Naive Bayes Training ===")
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

# ==== 9. Demo trên câu mới ====
samples = [
    "Xin chào, bạn có khỏe không?",       # Vietnamese (chưa nằm trong tập 10 ngôn ngữ)
    "Hello, how are you?",                 # English
    "Bonjour, je suis étudiant.",          # French
    "Guten Morgen, wie geht es dir?",      # German
    "สวัสดีครับ คุณทำอะไรอยู่?",            # Thai (ngoài tập huấn luyện)
    "Hola, ¿cómo estás?",                  # Spanish
    "Ciao, come stai?",                     # Italian
    "Olá, como você está?",                 # Portuguese
    "Привет, как дела?",                    # Russian
    "こんにちは、お元気ですか？"             # Japanese (ngoài tập huấn luyện)
]

print("\n=== Demo Predictions (LightGBM) ===")
for s in samples:
    print(s, " → ", lgbm_model.predict([s])[0])

print("\n=== Demo Predictions (Naive Bayes) ===")
for s in samples:
    print(s, " → ", nb_model.predict([s])[0])


Số mẫu mỗi ngôn ngữ: 437195


/tmp/ipython-input-488964916.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_count, random_state=42))


lang
ber    437195
deu    437195
eng    437195
epo    437195
fra    437195
ita    437195
kab    437195
por    437195
rus    437195
tur    437195
Name: count, dtype: int64
Balanced shape: (4371950, 3)

=== LightGBM Training ===
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2053.809883 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 333311
[LightGBM] [Info] Number of data points in the train set: 3497560, number of used features: 19995
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training from score -2.302585
[LightGBM] [Info] Start training

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Accuracy: 0.9452807099806723
              precision    recall  f1-score   support

         ber       0.83      0.79      0.81     87439
         deu       1.00      0.99      1.00     87439
         eng       1.00      0.99      0.99     87439
         epo       0.98      0.99      0.99     87439
         fra       0.99      0.98      0.99     87439
         ita       0.99      0.97      0.98     87439
         kab       0.81      0.81      0.81     87439
         por       0.99      0.98      0.99     87439
         rus       1.00      0.96      0.98     87439
         tur       0.86      0.99      0.92     87439

    accuracy                           0.95    874390
   macro avg       0.95      0.95      0.95    874390
weighted avg       0.95      0.95      0.95    874390


=== Naive Bayes Training ===
Naive Bayes Accuracy: 0.941956106542847
              precision    recall  f1-score   support

         ber       0.71      0.80      0.75     87439
         deu       1.00 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Bonjour, je suis étudiant.  →  fra
Guten Morgen, wie geht es dir?  →  deu


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


สวัสดีครับ คุณทำอะไรอยู่?  →  tur
Hola, ¿cómo estás?  →  por
Ciao, come stai?  →  ita
Olá, como você está?  →  por
Привет, как дела?  →  rus
こんにちは、お元気ですか？  →  tur

=== Demo Predictions (Naive Bayes) ===
Xin chào, bạn có khỏe không?  →  ber
Hello, how are you?  →  eng
Bonjour, je suis étudiant.  →  fra
Guten Morgen, wie geht es dir?  →  deu
สวัสดีครับ คุณทำอะไรอยู่?  →  ber
Hola, ¿cómo estás?  →  por
Ciao, come stai?  →  ita
Olá, como você está?  →  por
Привет, как дела?  →  rus
こんにちは、お元気ですか？  →  ber


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# ==== 8. Lưu pipeline lên Drive ====
import joblib
save_path = "/content/drive/MyDrive/lbgm_10langs.pkl"
joblib.dump(lgbm_model, save_path)
print("Saved model to:", save_path)
joblib.dump(nb_model, save_path)
print("Saved nb_model to:", save_path)

Saved model to: /content/drive/MyDrive/lbgm_10langs.pkl
Saved nb_model to: /content/drive/MyDrive/lbgm_10langs.pkl


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lngmnhlinh/vietnamese-lao")

print("Path to dataset files:", path)

100%|██████████| 29.0M/29.0M [00:00<00:00, 156MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/lngmnhlinh/vietnamese-lao/versions/2


In [ ]:
import os

dataset_path = "/root/.cache/kagglehub/datasets/lngmnhlinh/vietnamese-lao/versions/2/Vietnamese_Lao/VLSP2023/Train"

files = os.listdir(dataset_path)
print(files)


['train2023.vi', 'train2023.lo']


In [ ]:
import pandas as pd

# Đường dẫn tới file
vi_path = "/root/.cache/kagglehub/datasets/lngmnhlinh/vietnamese-lao/versions/2/Vietnamese_Lao/VLSP2023/Train/train2023.vi"
lo_path = "/root/.cache/kagglehub/datasets/lngmnhlinh/vietnamese-lao/versions/2/Vietnamese_Lao/VLSP2023/Train/train2023.lo"

# Đọc từng file, mỗi dòng là một câu
with open(vi_path, "r", encoding="utf-8") as f:
    vi_lines = f.read().splitlines()

with open(lo_path, "r", encoding="utf-8") as f:
    lo_lines = f.read().splitlines()

# Tạo DataFrame
df_vi = pd.DataFrame({"text": vi_lines, "lang": "vie"})
df_lo = pd.DataFrame({"text": lo_lines, "lang": "lao"})

# Ghép lại
df_cls = pd.concat([df_vi, df_lo]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_cls.head())
print("Số lượng câu mỗi ngôn ngữ:")
print(df_cls.lang.value_counts())


                                                text lang
0  14 ແລະ ເວລາ ນັ້ນ ການ ພິພາກສາ ຂອງ ພຣະຜູ້ ບໍລິສຸ...  lao
1                                     Cambridge, Anh  vie
2  ຊອກຫາສໍາລັບເຄື່ອງປັບອາກາດຫລ້າສຸດແລະຜະລິດຕະພັນລ...  lao
3  Lên đến 30 giây của một kịch bản tùy chỉnh với...  vie
4  Hầu hết chúng ta đều trải qua những thời kỳ tr...  vie
Số lượng câu mỗi ngôn ngữ:
lang
lao    100000
vie    100000
Name: count, dtype: int64


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import lightgbm as lgb

# ==== Load dữ liệu ==== #
with open(vi_path, "r", encoding="utf-8") as f:
    vi_lines = f.read().splitlines()

with open(lo_path, "r", encoding="utf-8") as f:
    lo_lines = f.read().splitlines()

df_vi = pd.DataFrame({"text": vi_lines, "lang": "vie"})
df_lo = pd.DataFrame({"text": lo_lines, "lang": "lao"})
df_cls = pd.concat([df_vi, df_lo]).sample(frac=1, random_state=42).reset_index(drop=True)

# ==== Train-test split ==== #
X_train, X_test, y_train, y_test = train_test_split(
    df_cls["text"], df_cls["lang"],
    test_size=0.2, stratify=df_cls["lang"], random_state=42
)

# ==== TF-IDF mới ==== #
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ==== Model 1: LightGBM ==== #
clf_lgb = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=2,
    boosting_type="gbdt",
    n_estimators=200,
    learning_rate=0.1,
    num_leaves=64,
    random_state=42
)
clf_lgb.fit(X_train_tfidf, y_train)
y_pred_lgb = clf_lgb.predict(X_test_tfidf)

print("=== LightGBM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lgb))
print(classification_report(y_test, y_pred_lgb))

# ==== Model 2: Naive Bayes ==== #
clf_nb = MultinomialNB()
clf_nb.fit(X_train_tfidf, y_train)
y_pred_nb = clf_nb.predict(X_test_tfidf)

print("\n=== Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

# ==== Demo ==== #
samples = [
    "Tôi đang học tiếng Việt.",
    "ສະບາຍດີ, ເຈົ້າສຸກດີບໍ?"
]

print("\n--- Demo Prediction ---")
for s in samples:
    print(s, " → LGB:", clf_lgb.predict(vectorizer.transform([s]))[0],
          " | NB:", clf_nb.predict(vectorizer.transform([s]))[0])


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 94.160269 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811598
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 24343
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Info] Start training from score -0.693147


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== LightGBM ===
Accuracy: 0.986475
              precision    recall  f1-score   support

         lao       0.98      1.00      0.99     20000
         vie       1.00      0.97      0.99     20000

    accuracy                           0.99     40000
   macro avg       0.99      0.99      0.99     40000
weighted avg       0.99      0.99      0.99     40000


=== Naive Bayes ===
Accuracy: 0.992675
              precision    recall  f1-score   support

         lao       0.99      1.00      0.99     20000
         vie       1.00      0.99      0.99     20000

    accuracy                           0.99     40000
   macro avg       0.99      0.99      0.99     40000
weighted avg       0.99      0.99      0.99     40000


--- Demo Prediction ---
Tôi đang học tiếng Việt.  → LGB: vie  | NB: vie
ສະບາຍດີ, ເຈົ້າສຸກດີບໍ?  → LGB: lao  | NB: lao


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
